# Kaggle 最小训练模板

按顺序运行下列单元格。训练项目必须是一个独立 Git 仓库，并包含：

- 导出 `build_experiment(config)` 的 Python 模块；
- 严格 schema v1 训练配置；
- 项目自身需要的第三方依赖。

Kaggle 需要在 Secrets 中提供配置引用的 AList 和企业微信键。`train` 会先自动预检，
省略恢复参数时由 CLI 内部自动处理本地或远端检查点。退出码 `75` 表示预算保护暂停，
不是失败；在新 Session 重跑训练单元格即可继续。


In [ ]:
from pathlib import Path

# 替换为你的 dl_helper 仓库和固定版本。
DL_HELPER_REPO_URL = "https://github.com/lhiqwj173/dl_helper.git"
DL_HELPER_REF = "master"

# 替换为你的训练项目仓库和固定版本。
TRAINING_REPO_URL = "https://github.com/your-account/my-training-project.git"
TRAINING_REF = "main"

PROJECT_DIR = Path("/kaggle/working/my-project")
CONFIG_PATH = PROJECT_DIR / "configs" / "kaggle.yaml"
EXPERIMENT = "my_experiment:build_experiment"
RUN_ID = "my-project-v1"

for name, value in (
    ("DL_HELPER_REPO_URL", DL_HELPER_REPO_URL),
    ("DL_HELPER_REF", DL_HELPER_REF),
    ("TRAINING_REPO_URL", TRAINING_REPO_URL),
    ("TRAINING_REF", TRAINING_REF),
    ("RUN_ID", RUN_ID),
):
    if not value or any(character.isspace() for character in value):
        raise ValueError(f"{name} 必须是非空且不含空白的字符串")
    if "your-account" in value:
        raise ValueError(f"请先把 {name} 替换为实际值")

module_name, separator, factory_name = EXPERIMENT.partition(":")
if not separator or not module_name or not factory_name:
    raise ValueError("EXPERIMENT 必须使用 module:function 格式")
if any(not part or part == "." or ".." in part for part in module_name.split(".")):
    raise ValueError(f"Experiment 模块名无效: {module_name!r}")

print("dl_helper:", DL_HELPER_REF)
print("training project:", TRAINING_REF)
print("experiment:", EXPERIMENT)
print("run id:", RUN_ID)


In [ ]:
import os
import subprocess
import sys


def run_checked(argv, *, cwd=None):
    print("+", " ".join(argv))
    process = subprocess.run(argv, cwd=cwd, text=True, encoding="utf-8")
    if process.returncode != 0:
        raise RuntimeError(f"命令失败，退出码 {process.returncode}: {' '.join(argv)}")
    return process


def clone_fixed(repo_url, ref, target):
    if target.exists():
        raise RuntimeError(f"目标目录已存在，请新建 Kaggle Session 后重试: {target}")
    run_checked(["git", "clone", repo_url, str(target)])
    run_checked(["git", "checkout", ref], cwd=target)
    head = run_checked(["git", "rev-parse", "HEAD"], cwd=target).stdout.strip()
    expected = run_checked(
        ["git", "rev-parse", f"{ref}^{{commit}}"], cwd=target
    ).stdout.strip()
    if head.lower() != expected.lower():
        raise RuntimeError(f"checkout HEAD 不匹配: {head} != {expected}")
    return target


DL_HELPER_DIR = clone_fixed(
    DL_HELPER_REPO_URL,
    DL_HELPER_REF,
    Path("/kaggle/working/dl-helper"),
)
os.environ["DL_HELPER_GIT_REPO"] = DL_HELPER_REPO_URL
os.environ["DL_HELPER_REPO_DIR"] = str(DL_HELPER_DIR)
run_checked(
    [sys.executable, str(DL_HELPER_DIR / "envs" / "kaggle_bootstrap.py")],
    cwd=DL_HELPER_DIR,
)


In [ ]:
clone_fixed(TRAINING_REPO_URL, TRAINING_REF, PROJECT_DIR)

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"训练配置不存在: {CONFIG_PATH}")

module_path = PROJECT_DIR.joinpath(*module_name.split("."))
module_candidates = (
    module_path.with_suffix(".py"),
    module_path / "__init__.py",
)
if not any(path.is_file() for path in module_candidates):
    searched = " 或 ".join(str(path) for path in module_candidates)
    raise FileNotFoundError(f"Experiment 模块不存在于项目目录内: {searched}")

print("project dir:", PROJECT_DIR)
print("config:", CONFIG_PATH)


In [ ]:
train_argv = [
    sys.executable,
    "-m",
    "dl_helper.training.cli",
    "train",
    "--project-dir", str(PROJECT_DIR),
    "--config", str(CONFIG_PATH),
    "--experiment", EXPERIMENT,
    "--run-id", RUN_ID,
]

train_process = subprocess.run(train_argv, cwd=DL_HELPER_DIR, text=True, encoding="utf-8")
if train_process.returncode == 75:
    print("训练因 Kaggle 预算保护暂停；新 Session 重跑本单元格即可自动继续。")
elif train_process.returncode != 0:
    raise RuntimeError(f"训练失败，退出码: {train_process.returncode}")
else:
    print("训练完成。")
